# **Анализ конкурентов**

In [68]:
# устанавливаем библиотеки
!pip install -q requests beautifulsoup4 lxml playwright pandas numpy plotly dash nest_asyncio
!playwright install chromium

import re
import time
import uuid
import json
import asyncio
import requests
import numpy as np
import pandas as pd
import nest_asyncio

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from playwright.async_api import async_playwright

import plotly.express as px
import plotly.graph_objects as go

nest_asyncio.apply()

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)  # утключаем предупреждения о небезопасных HTTPS-соединениях

In [69]:
# вручную составляем список конкурентов
# буду писать комментарии по каждой фирме (телеграм и тип по итогу не понадобились)
competitors = [
    {
        "name": "UniPage",
        "website": "https://www.unipage.net/ru",   #страница с услугами https://www.unipage.net/ru/services
        "telegram": "https://t.me/unipage",        # нет отзывов
        "type": "Платформа / агентство"
    },
    {
        "name": "Allterra Education",
        "website": "https://allterra.ru",   #нет открытых цен, но есть страница услуги
        "telegram": "https://t.me/allterra_education",  # 1 отзыв
        "type": "Агентство"
    },
    {
        "name": "STAR Academy",
        "website": "https://www.staracademy.ru",   #нет открытых цен, но есть страница услуги
        "telegram": "",
        "type": "Агентство"
    },
    {
        "name": "IQ Consultancy",
        "website": "https://iqconsultancy.ru",   #нет открытых цен
        "telegram": "",
        "type": "Агентство"
    },
    {
        "name": "Educate Online",
        "website": "https://educate-online.ru",  #есть цены в долларах https://educate-online.ru/vo
        "telegram": "",
        "type": "EdTech / агентство"
    },
    {
        "name": "Academic Advising Center",
        "website": "https://academconsult.ru",  #много разных услуг с ценами, но немного убитый сайт https://academconsult.ru/uslugi/
        "telegram": "",
        "type": "Агентство"
    },
    {
        "name": "Students International",
        "website": "https://studinter.ru",  #нет открытых цен
        "telegram": "",
        "type": "Агентство"
    },
    {
        "name": "Itec",
        "website": "https://itecgroup.ru",  #нет открытых цен, но красивый и многоступенчатый сайт
        "telegram": "",
        "type": "Агентство"
    },
    {
        "name": "EDUCATION INDEX",
        "website": "https://educationindex.ru",  #нет открытых цен
        "telegram": "",
        "type": "Платформа / агентство"
    },
    {
        "name": "Global Dialog",
        "website": "https://www.globaldialog.ru",  #есть цены и страница услуги https://www.globaldialog.ru/company/services/ (текст сплошной)
        "telegram": "",
        "type": "Агентство"
    },
    # {
    #     "name": "London Gates Education Group",
    #     "website": "https://londongates.org",  #больше школа доп образования. у них есть нужная услуга, но далеко не основная. Не основной приоритет
    #     "telegram": "",
    #     "type": "Агентство"
    # },
    {
        "name": "Smapse Education",
        "website": "https://smapse.ru",    #нет открытых цен, но есть страница услуги https://smapse.ru/uslugi-i-tseni/
        "telegram": "",
        "type": "Платформа / агентство"
    },
    {
        "name": "DEC Education",
        "website": "https://dec-edu.com",  #нет открытых цен (полуиностранный сайт)
        "telegram": "",
        "type": "Агентство"
    },
    {
        "name": "EduTravel",
        "website": "https://edutravel.ru",   #есть цены, но не на одной странице. Сначала нажимаешь на услуги, потом в меню выбираешь нужную страницу
        "telegram": "",
        "type": "Агентство"
    },
    {
        "name": "StudyLab",
        "website": "https://studylab.ru",  #нет открытых цен
        "telegram": "",
        "type": "Агентство"
    },
    {
        "name": "LinguaTrip",
        "website": "https://linguatrip.com/ru",   #нет открытых цен. Услуги по поступлению на странице https://linguatrip.com/ru/higher_education/
        "telegram": "",
        "type": "EdTech / языковое обучение"
    }
]

После просмотра сайтов оказалось, что очень мало где пишут цены. Это делает поиск цен в парсинге бесполезным. Других источников с ценами тоже найти не получилось.

На практике происходит так, что студент заинтересованный в приобретении подобных услуг просто записывается на консультации в нескольких фирм и после этого сравнивает цены, которые они называют.

Поэтому в данном случае в анализе придется обойтись без цен (в дальнейшем при развитии проекта можно будет узнать их вручную)

Для полноты анализа хотим разделить конкурентов на группы и посчитать метрики для каждой из них.
Потенциальные возможности деления
- по направлениям обучения (с какими направлениями рабоатает агенство)
- по странам

На сайтах обычно не пишут про направления, поэтому оставим этот метод для парсинга телеграма (можно будет оценить наиболее попутярные направления там)

Что касается стран, то их на сайтах найти возможно

Также хотим посмотреть, какие виды услуг предлагает каждое агенство (есть ли языковые курсы, подготовка к экзаменам. Или есть и более узконаправленные агенства)

In [70]:
#используем словари чтобы искать в тексте сайтов ключевые слова. С помощью этого можем понять с какими странами работает агенство, какие услуги предлагает
# направления обучения (по итогу не понадобились)
directions = {
    "STEM": [
        "computer science", "data science", "engineering", "mathematics",
        "physics", "artificial intelligence", "machine learning",
        "программирование", "инженерия", "математика", "информатика",
        "искусственный интеллект", "машинное обучение", "технологии", "stem",
        " it ", "айти"
    ],
    "Бизнес и менеджмент": [
        "mba", "business", "management", "finance", "economics",
        "marketing", "entrepreneurship", "бизнес", "менеджмент",
        "финансы", "экономика", "маркетинг", "предпринимательство",
        "управление", "business school"
    ],
    "Гуманитарные и социальные науки": [
        "psychology", "sociology", "political science", "philosophy",
        "history", "international relations", "психология", "социология",
        "политология", "философия", "история", "международные отношения",
        "социальные науки", "гуманитарные науки"
    ],
    "Искусство и дизайн": [
        "fine arts", "graphic design", "architecture", "fashion", "film",
        "animation", "illustration", "дизайн", "архитектура", "искусство",
        "мода", "кино", "анимация", "иллюстрация", "арт", "creative"
    ],
    "Медицина и здравоохранение": [
        "medicine", "dentistry", "pharmacy", "nursing", "public health",
        "biomedicine", "медицина", "стоматология", "фармация",
        "здравоохранение", "медицинский", "биомедицина"
    ],
    "Право": [
        "law", "legal", "jurisprudence", "llm", "право",
        "юриспруденция", "правоведение", "юридический"
    ],
    "Языки и лингвистика": [
        "linguistics", "translation", "languages", "tesol", "ielts",
        "toefl", "cambridge english", "лингвистика", "перевод",
        "языки", "английский язык", "иностранный язык"
    ],
    "Образование и педагогика": [
        "education", "teaching", "pedagogy", "педагогика",
        "образование", "преподавание", "teacher"
    ]
}


countries = {
    "США": ["сша", "usa", "united states", "america", "америка", "u.s."],
    "Великобритания": ["великобритания", "uk", "united kingdom", "англия", "britain"],
    "Германия": ["германия", "germany", "deutschland"],
    "Канада": ["канада", "canada"],
    "Австралия": ["австралия", "australia"],
    "Нидерланды": ["нидерланды", "netherlands", "голландия", "holland"],
    "Швейцария": ["швейцария", "switzerland"],
    "Франция": ["франция", "france"],
    "Италия": ["италия", "italy"],
    "Испания": ["испания", "spain"],
    "Китай": ["китай", "china"],
    "ОАЭ": ["оаэ", "uae", "dubai", "дубай", "эмираты"],
    "Чехия": ["чехия", "czech", "czech republic", "прага", "prague"],
    "Венгрия": ["венгрия", "hungary"],
    "Польша": ["польша", "poland"],
    "Австрия": ["австрия", "austria"],
    "Ирландия": ["ирландия", "ireland"],
    "Швеция": ["швеция", "sweden"],
    "Финляндия": ["финляндия", "finland"],
    "Дания": ["дания", "denmark"],
    "Норвегия": ["норвегия", "norway"],
    "Сингапур": ["сингапур", "singapore"],
    "Южная Корея": ["южная корея", "korea", "south korea"],
    "Япония": ["япония", "japan"]
}


services = {
    "Полное сопровождение": [
        "полное сопровождение", "комплексное сопровождение", "сопровождение под ключ",
        "поступление под ключ", "полный пакет", "full support",
        "complete package", "complete admission support"
    ],
    "Частичное сопровождение": [
        "частичное сопровождение", "отдельная услуга", "отдельные услуги",
        "помощь с подачей", "помощь с заявкой", "application support",
        "partial support"
    ],
    "Подбор университетов": [
        "подбор университетов", "подбор вузов", "подбор программ",
        "выбор университета", "выбор программы", "university selection",
        "program selection", "shortlist"
    ],
    "Разовая консультация": [
        "консультация", "индивидуальная консультация", "разовая консультация",
        "час консультации", "consultation", "consulting session", "бесплатная консультация"
    ],
    "Документы / эссе": [
        "мотивационное письмо", "эссе", "personal statement",
        "statement of purpose", "sop", "recommendation letter",
        "рекомендательное письмо", "резюме", "cv", "портфолио",
        "проверка документов", "подготовка документов", "essay review"
    ],
    "Визовая поддержка": [
        "студенческая виза", "учебная виза", "виза для учебы", "виза для учёбы",
        "визовая поддержка", "оформление визы", "student visa",
        "visa support", "visa application"
    ],
    "Языковая подготовка": [
        "ielts", "toefl", "gmat", "gre", "sat", "act",
        "языковая подготовка", "подготовка к экзамену",
        "language preparation", "test preparation"
    ]
}

In [71]:
#класс парсер сайтов
class OfficialSiteScraper:

    def __init__(
        self,
        max_pages_per_site = 6,  #максимум страниц, которые читаем у одного конкурента
        timeout = 15,   #сколько секунд ждём ответа сайта (чтобы не ждать бесконечно долго, если сайт упал, игнорирует наш запрос и тд)
        pause_seconds = 1.0,  #пауза между запросами (чтобы сайт не заблокировал запрос к нему)
        min_text_length = 250  #минимальная длина текста, чтобы считать сайт успешно спарсенным (вообще я уже поудаляла сайты, которые не парсились, но все равно оставим это)
    ):

        self.max_pages_per_site = max_pages_per_site
        self.timeout = timeout
        self.pause_seconds = pause_seconds
        self.min_text_length = min_text_length

        # заголовки имитируют обычный браузер и это зашишает запросы от участи быть отклоненными сайтами
        self.headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0 Safari/537.36"
            ),
            "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
        }

        # ключевые слова для выбора полезных внутренних страниц
        # нам нужны страницы про услуги, страны, обучение, поступление
        self.important_page_keywords = [ "services", "service", "услуги", "uslugi", "countries", "country", "strany", "страны", "education", "study", "обучение", "ucheba", "учеба", "учёба",
            "admission", "apply", "application", "поступление", "postuplenie", "higher", "university", "universities", "vuz", "вузы", "университет","visa", "виза", "about", "company", "о-нас", "onas"]

    # очистка от лишних пробелов (было много пробелов, сделали один) и переносов строк
    def clean_text(self, text):
        if not isinstance(text, str):
            return ""
        return re.sub(r"\s+", " ", text).strip()

    # приводим url ссылки в единый вид (чтобы обрабатывать внутри несколько страниц и например не заходить на одну и ту же страницу несколько раз)
    def normalize_url(self, url):
        if not url:
            return ""

        url = url.split("#")[0].strip()
        # зачем это нужно. url на сайтах выглядит как сслылканастраницу#скольконужнопрокрутить. Нам прокрутка не интересна, поэтому берем только ссылку на сайт

        # убираем слэш в конце, если он вдруг есть
        if url.endswith("/"):
            url = url[:-1]
        return url

    # достаем домен из url, чтобы отличить внутренние ссылки от внешних (нам интересны только внутренние)
    def get_domain(self, url):
        try:
            return urlparse(url).netloc.replace("www.", "")
        except Exception:  # ловим ошибку если вдруг прилетело что-то не то
            return ""
    # проверяем является ли ссылка внутренней
    def is_internal_url(self, base_url, candidate_url):
        return self.get_domain(base_url) == self.get_domain(candidate_url)

    # пауза между запросами чтобы не заблочили
    def pause(self):
        time.sleep(self.pause_seconds)

    # ищем ключевые слова в тексте, если хотя бы одно ключевое слово найдено - добавляем категорию в результат
    def find_matches(self, text, keyword_dict):
        text_lower = text.lower()
        found = []

        for category, keywords in keyword_dict.items():
            if any(keyword.lower() in text_lower for keyword in keywords):
                found.append(category)

        return sorted(list(set(found)))

    # отправляем запросы
    def fetch_with_requests(self, url):
        try:
            response = requests.get(
                url,
                headers=self.headers,
                timeout=self.timeout,
                allow_redirects=True,
                verify=False
            )

            if response.status_code >= 400:  # если он 200, то все нормально, а если 400+ то это ошибки. Выводим их (я заранее поудаляла все проблемные сайты, но пусть останется тут)
                return "", f"HTTP {response.status_code}", "requests" # шаблон для вывода, распишу позднее

            response.encoding = response.apparent_encoding # метод библиотеки requests
            # проблема с русскими сайтами. Часто может попадаться старая кодировка и выведется что-то неадекватное. Эта формула приводит все в читаемый текст

            return response.text, "", "requests" # шаблон: html текста, ошибка (если есть, иначе пустая строка), что использовали (если несколько источников)

        except Exception as e:
            return "", str(e), "requests"  # при ошибке выводим ее по шаблону

    # если прошлый метод не справился, используем этот (парсинг динамических сайтов)
    # умеет работать с java, в отличие от прошлого
    async def fetch_with_playwright_async(self, url):
        try:
            async with async_playwright() as p:  # экономия времени через asyncio
                browser = await p.chromium.launch(headless=True)  #headless=True запускает браузер в фоновом режиме, await ожидание (отдает флажок другим задачам)

                context = await browser.new_context(    # маскировка
                    user_agent=self.headers["User-Agent"],
                    locale="ru-RU",
                    viewport={"width": 1366, "height": 900}  # как будто монитор
                )

                page = await context.new_page()  # как будто создаем новую вкладку

                await page.goto(
                    url,
                    wait_until="domcontentloaded",   # не ждем пока загрузятся картинки и большие данные, заканчиваем сразу как загрузится базовый текст и каркас
                    timeout=30000   #если долго нет ответа прерываем
                )

                # немного ждем, чтобы динамические штуки смогли прогрузиться
                await page.wait_for_timeout(2000)

                html = await page.content()  # получаем код страницы

                await browser.close() # закрываем и освобождаем оперативную память

                return html, "", "playwright"

        except Exception as e:
            return "", str(e), "playwright"

    # реализация прошлых методов. Сначала requests, потом playwright и выводим результат
    def fetch_html(self, url):
        html, error, method = self.fetch_with_requests(url)

        text = self.extract_text_from_html(html)  # метод задан ниже

        if len(text) < self.min_text_length:  # если что-то пошло не так и результат подозрительно маленький
            try:
                html_pw, error_pw, method_pw = asyncio.get_event_loop().run_until_complete(
                    self.fetch_with_playwright_async(url)
                )  # нельзя просто так в вызвать асинхронную функцию в обычной, поэтому создается коридор

                text_pw = self.extract_text_from_html(html_pw) # получаем новый результат вторым методом

                if len(text_pw) > len(text):  # сравниваем. если новый лучше, то берем его
                    return html_pw, error_pw, method_pw

            except Exception as e:
                if not error:
                    error = f"Playwright fallback failed: {e}"

        return html, error, method

    # преварщаем html в текст
    def extract_text_from_html(self, html):
        if not html:
            return ""

        soup = BeautifulSoup(html, "lxml")  # выбираем lxml потому что быстрый и хорошо работает с грязным кодом с сайтов

        # удаляем технические теги
        for tag in soup(["script", "style", "noscript", "svg"]): # убирает код java, оформление текста (цвета), мусор (что показывается если отключен java в браузере), иконки и картинки
            tag.extract()

        return self.clean_text(soup.get_text(" ")) # собирает все что осталось в текст (get_text - внутренний метод bs, который удаляет теги), отправляет его на очистку в ранее написанный метод и возвращает

    # разбор страницы
    def parse_page(self, url, html):

        # шаблон результата
        result = {
            "url": url,
            "title": "",
            "meta_description": "",   # краткое описание страницы для поисковиков
            "text": "",
            "internal_links": []
        }

        if not html:
            return result

        soup = BeautifulSoup(html, "lxml")

        # заголовок страницы
        if soup.title and soup.title.text:
            result["title"] = self.clean_text(soup.title.text)

        # meta описание
        meta = soup.find("meta", attrs={"name": "description"})
        if meta and meta.get("content"):
            result["meta_description"] = self.clean_text(meta.get("content"))

        # текст страницы
        result["text"] = self.extract_text_from_html(html)

        internal_links = []

        # внутренние ссылки
        for a in soup.find_all("a", href=True):  # а - обозначение ссылок, href=True - проверка, что ссылка там точно есть
            href = a.get("href", "") # выводим ссылку, иначе ничего ("" дефолтное значение)
            anchor = self.clean_text(a.get_text(" ")) # заголовок страницы (шаблон: <a href = ...>Название страницы</a>)

            abs_url = self.normalize_url(urljoin(url, href)) # urljoin приклеивает новую ссылку в текущей тк часто пишут сокращенно. normalize_url сами писали ранее

            if not abs_url.startswith("http"):  # если это не страница, то дропаем итерацию
                continue

            # пропускаем ссылки на внешние программы (тоже дропаем итерацию)
            if any(x in abs_url.lower() for x in ["mailto:", "tel:", "wa.me", "whatsapp"]):
                continue

            # нас интересуют только внутренние ссылки
            if self.is_internal_url(url, abs_url):
                internal_links.append({
                    "url": abs_url,
                    "anchor": anchor
                })

        result["internal_links"] = internal_links
        return result

    # мы не хотим смотреть все ссылки подряд, поэтому нужно оценить их полезность
    # немного странно, но мне нравится начисление баллов))
    def score_link(self, link):
        url = link.get("url", "").lower()
        anchor = link.get("anchor", "").lower()
        combined = url + " " + anchor # ищем ключевые слова сразу во всем

        score = 0
        for keyword in self.important_page_keywords:
            if keyword.lower() in combined:
                score += 10

        # пропускаем файлы и картинки
        bad_extensions = [".pdf", ".jpg", ".jpeg", ".png", ".webp", ".doc", ".docx", ".xls", ".xlsx"]
        if any(url.endswith(ext) for ext in bad_extensions):
            score -= 1000

        # очень длинные url нам точно не нужны
        if len(url) > 180:
            score -= 100

        return score

    # выбираем что будем парсить (главная страница + наиболее полезные из остальных)
    def choose_pages_to_parse(self, home_url, home_page):
        # home_page - уже спарсенная с помощью parse_page страница (в этой переменной лежит шаблон result с данными)
        selected_urls = [self.normalize_url(home_url)] # начинаем с домашней страницы

        links = home_page.get("internal_links", [])

        # убираем дубликаты
        unique_links = {}
        for link in links:
            url = self.normalize_url(link["url"])
            if url and url not in unique_links:
                unique_links[url] = link

        # сортируем по полезности
        scored_links = sorted(
            unique_links.values(),
            key = lambda x: self.score_link(x),
            reverse = True
        )

        # добавляем только ссылки с положительной оценкой
        for link in scored_links:
            if len(selected_urls) >= self.max_pages_per_site:
                break
            if self.score_link(link) <= 0:
                continue
            url = self.normalize_url(link["url"])
            if url not in selected_urls: # можно еще раз не проверять, но пусть будет
                selected_urls.append(url)

        return selected_urls

    # парсим одного конкурента
    def parse_one(self, competitor):
        name = competitor["name"]
        home_url = self.normalize_url(competitor["website"])

        errors = []
        parsed_pages = []

        # загружаем главную страницу
        home_html, home_error, home_method = self.fetch_html(home_url)

        if home_error:
            errors.append(f"home: {home_error}")

        home_page = self.parse_page(home_url, home_html)

        # выбираем полезные страницы
        pages_to_parse = self.choose_pages_to_parse(home_url, home_page)

        # парсим выбранные страницы
        for idx, page_url in enumerate(pages_to_parse, start=1): # нумеруем
            print(f"    [{idx}/{len(pages_to_parse)}] {page_url}")

            html, error, method = self.fetch_html(page_url)

            if error:
                errors.append(f"{page_url}: {error}")

            page_data = self.parse_page(page_url, html)

            # общая инфа про то, что получилось
            parsed_pages.append({
                "url": page_url,
                "method": method,
                "title": page_data["title"],
                "text_length": len(page_data["text"]),
                "text": page_data["text"]
            })

            self.pause()

        # объединяем текст всех страниц
        combined_text = " ".join([p["text"] for p in parsed_pages])
        combined_text = self.clean_text(combined_text)

        # ищем страны и услуги
        countries_found = self.find_matches(combined_text, countries)
        services_found = self.find_matches(combined_text, services)

        # считаем сколько чего
        countries_count = len(countries_found)
        services_count = len(services_found)

        fetch_status = "ok" if len(combined_text) >= self.min_text_length else "error"

        result = {
            "name": name,
            "website": home_url,
            "pages_parsed": len(parsed_pages),
            "total_text_length": len(combined_text),
            "errors": errors,
            "countries_found": countries_found,
            "services_found": services_found,
            "countries_count": countries_count,
            "services_count": services_count,
        }

        return result

    # парсим всех
    def parse_all(self, competitors):
        results = []

        for i, competitor in enumerate(competitors, start=1):
            # наблядаем за процессом
            print("=" * 80)
            print(f"[{i}/{len(competitors)}] Парсим сайт: {competitor['name']}")
            print(f"URL: {competitor['website']}")

            try:
                result = self.parse_one(competitor)

            except Exception as e:
                # eсли один сайт упал не останавливаем весь проект (выводим шаблон для ошибки) опять же я все проблемные сайты уже убрала, в результате мы их не увидим
                result = {
                    "name": competitor.get("name", ""),
                    "website": competitor.get("website", ""),
                    "pages_parsed": 0,
                    "total_text_length": 0,
                    "errors": [str(e)],
                    "countries_found": [],
                    "services_found": [],
                    "countries_count": 0,
                    "services_count": 0,
                }

            results.append(result)

            print("Итог:")
            print(f"Страниц спарсено: {result['pages_parsed']}")
            print(f"Длина текста: {result['total_text_length']}")
            print(f"Страны: {result['countries_found']}")
            print(f"Услуги: {result['services_found']}")

            if result["errors"]:
                print(f"  Ошибки: {result['errors'][:2]}")
            print()

        return results

In [72]:
site_scraper = OfficialSiteScraper(
    max_pages_per_site=6,
    timeout=15,
    pause_seconds=1.0,
    min_text_length=250
)

site_results = site_scraper.parse_all(competitors)
site_df = pd.DataFrame(site_results)  # закидываем все в таблицу

[1/15] Парсим сайт: UniPage
URL: https://www.unipage.net/ru
    [1/6] https://www.unipage.net/ru
    [2/6] https://www.unipage.net/ru/services
    [3/6] https://www.unipage.net/ru/study_countries
    [4/6] https://www.unipage.net/ru/universities_countries
    [5/6] https://www.unipage.net/ru/secondary_education_countries
    [6/6] https://www.unipage.net/ru/higher_education_countries
Итог:
Страниц спарсено: 6
Длина текста: 88239
Страны: ['Австралия', 'Австрия', 'Великобритания', 'Венгрия', 'Германия', 'Дания', 'Ирландия', 'Испания', 'Италия', 'Канада', 'Китай', 'Нидерланды', 'Норвегия', 'ОАЭ', 'Польша', 'США', 'Сингапур', 'Финляндия', 'Франция', 'Чехия', 'Швейцария', 'Швеция', 'Южная Корея', 'Япония']
Услуги: ['Документы / эссе', 'Подбор университетов', 'Полное сопровождение', 'Разовая консультация', 'Языковая подготовка']

[2/15] Парсим сайт: Allterra Education
URL: https://allterra.ru
    [1/6] https://allterra.ru
    [2/6] https://allterra.ru/company/about-the-company
    [3/6] http

In [73]:
# подготовка данных

# список в строку. Нужно только чтобы таблицы красиво смотрелись
def list_to_str(x):
    if isinstance(x, list):
        return ", ".join(map(str, x))
    return ""

# создаём копию основной таблицы (чтобы не портить основную)
analysis_df = site_df.copy()

analysis_df["Страны"] = analysis_df["countries_found"].apply(list_to_str)
analysis_df["Услуги"] = analysis_df["services_found"].apply(list_to_str)

# переименовываем колонки
analysis_df = analysis_df.rename(
    columns={
        "name": "Конкурент",
        "website": "Сайт",
        "pages_parsed": "Страниц обработано",
        "total_text_length": "Объём текста",
        "countries_count": "Число стран",
        "services_count": "Число услуг",
    }
)

main_table_cols = [
    "Конкурент",
    "Сайт",
    "Число стран",
    "Число услуг",
    "Страны",
    "Услуги",
    "Страниц обработано",
    "Объём текста",
]

main_table_df = analysis_df[main_table_cols].copy()
display(main_table_df)

,Конкурент,Сайт,Число стран,Число услуг,Страны,Услуги,Страниц обработано,Объём текста
0,UniPage,https://www.unipage.net/ru,24,5,"Австралия, Австрия, Великобритания, Венгрия, Г...","Документы / эссе, Подбор университетов, Полное...",6,88239
1,Allterra Education,https://allterra.ru,1,1,Великобритания,Языковая подготовка,6,27718
2,STAR Academy,https://www.staracademy.ru,23,3,"Австралия, Австрия, Великобритания, Венгрия, Г...","Документы / эссе, Подбор университетов, Языков...",6,41083
3,IQ Consultancy,https://iqconsultancy.ru,17,4,"Австралия, Австрия, Великобритания, Германия, ...","Визовая поддержка, Документы / эссе, Разовая к...",6,70910
4,Educate Online,https://educate-online.ru,2,1,"Канада, США",Языковая подготовка,2,203312
5,Academic Advising Center,https://academconsult.ru,23,7,"Австралия, Австрия, Великобритания, Венгрия, Г...","Визовая поддержка, Документы / эссе, Подбор ун...",6,53313
6,Students International,https://studinter.ru,20,4,"Австралия, Австрия, Великобритания, Германия, ...","Документы / эссе, Подбор университетов, Полное...",6,104841
7,Itec,https://itecgroup.ru,19,3,"Австралия, Австрия, Великобритания, Венгрия, Г...","Визовая поддержка, Документы / эссе, Языковая ...",6,85153
8,EDUCATION INDEX,https://educationindex.ru,15,6,"Австралия, Великобритания, Германия, Дания, Ис...","Визовая поддержка, Документы / эссе, Подбор ун...",6,52744
9,Global Dialog,https://www.globaldialog.ru,16,6,"Австрия, Великобритания, Германия, Дания, Ирла...","Визовая поддержка, Документы / эссе, Подбор ун...",6,86032


In [74]:
# меторики по странам
country_rows = []

for _, row in site_df.iterrows():
    competitor_name = row["name"]
    found_countries = row.get("countries_found", [])

    if not isinstance(found_countries, list):
        continue

    for country in found_countries:
        country_rows.append({"Конкурент": competitor_name,"Страна": country})

country_mentions_df = pd.DataFrame(country_rows)

if len(country_mentions_df) > 0:
    country_metrics_df = (  # цепочка методов, которые выполняются друг за другом (поэтому с точками)
        country_mentions_df
        .groupby("Страна")   # группировка по странам
        .agg(Количество_конкурентов=("Конкурент", "nunique")) # если название одной страны встретилось на одном сайте много раз, считаем только 1 раз
        .reset_index()  #
        .sort_values("Количество_конкурентов", ascending=False) # сортировка по убыванию
    )
else:
    country_metrics_df = pd.DataFrame(columns=["Страна", "Количество_конкурентов"])

display(country_metrics_df)
# числа странные потому что отсортировали

,Страна,Количество_конкурентов
2,Великобритания,13
9,Канада,13
15,США,13
4,Германия,11
8,Италия,11
11,Нидерланды,11
7,Испания,11
18,Франция,11
1,Австрия,10
10,Китай,10


In [75]:
# метрики по услугам
service_rows = []

for _, row in site_df.iterrows():
    competitor_name = row["name"]
    found_services = row.get("services_found", [])

    if not isinstance(found_services, list):
        continue

    for service in found_services:
        service_rows.append({"Конкурент": competitor_name,"Услуга": service})

service_mentions_df = pd.DataFrame(service_rows)

# аналогично как в прошлой
if len(service_mentions_df) > 0:
    service_metrics_df = (
        service_mentions_df
        .groupby("Услуга")
        .agg(Количество_конкурентов=("Конкурент", "nunique"))
        .reset_index()
        .sort_values("Количество_конкурентов", ascending=False)
    )
else:
    service_metrics_df = pd.DataFrame(columns=["Услуга", "Количество_конкурентов"])

display(service_metrics_df)

,Услуга,Количество_конкурентов
6,Языковая подготовка,14
1,Документы / эссе,12
2,Подбор университетов,7
4,Разовая консультация,7
0,Визовая поддержка,6
3,Полное сопровождение,6
5,Частичное сопровождение,1


In [76]:
# таблица чтобы смотреть к какой фирмы есть какая услуга
service_matrix_rows = []

for _, row in site_df.iterrows():
    item = {"Конкурент": row["name"]}

    found_services = row.get("services_found", [])

    if not isinstance(found_services, list):
        found_services = []

    for service_name in services.keys():
        item[service_name] = 1 if service_name in found_services else 0

    service_matrix_rows.append(item)

service_matrix_df = pd.DataFrame(service_matrix_rows)

display(service_matrix_df)

,Конкурент,Полное сопровождение,Частичное сопровождение,Подбор университетов,Разовая консультация,Документы / эссе,Визовая поддержка,Языковая подготовка
0,UniPage,1,0,1,1,1,0,1
1,Allterra Education,0,0,0,0,0,0,1
2,STAR Academy,0,0,1,0,1,0,1
3,IQ Consultancy,0,0,0,1,1,1,1
4,Educate Online,0,0,0,0,0,0,1
5,Academic Advising Center,1,1,1,1,1,1,1
6,Students International,1,0,1,0,1,0,1
7,Itec,0,0,0,0,1,1,1
8,EDUCATION INDEX,1,0,1,1,1,1,1
9,Global Dialog,1,0,1,1,1,1,1


In [77]:
# то же самое, только со странами
country_matrix_rows = []
for _, row in site_df.iterrows():
    item = {"Конкурент": row["name"]}
    found_countries = row.get("countries_found", [])

    if not isinstance(found_countries, list):
        found_countries = []

    for country_name in countries.keys():
        item[country_name] = 1 if country_name in found_countries else 0

    country_matrix_rows.append(item)

competitor_country_matrix_df = pd.DataFrame(country_matrix_rows)
display(competitor_country_matrix_df)

,Конкурент,США,Великобритания,Германия,Канада,Австралия,Нидерланды,Швейцария,Франция,Италия,...,Польша,Австрия,Ирландия,Швеция,Финляндия,Дания,Норвегия,Сингапур,Южная Корея,Япония
0,UniPage,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
1,Allterra Education,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,STAR Academy,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,0,1,1,1
3,IQ Consultancy,1,1,1,1,1,1,1,1,1,...,0,1,0,1,0,1,0,1,0,0
4,Educate Online,1,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,Academic Advising Center,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,0,1
6,Students International,1,1,1,1,1,1,1,1,1,...,0,1,1,1,1,1,0,1,1,1
7,Itec,1,1,1,1,1,1,1,1,1,...,0,1,1,0,0,1,0,1,1,1
8,EDUCATION INDEX,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,1,0,1,0,0
9,Global Dialog,1,1,1,1,0,1,0,1,1,...,0,1,1,0,0,1,0,1,1,1


In [78]:
# популярность стран
fig_countries = px.bar(
    country_metrics_df.sort_values("Количество_конкурентов", ascending=True).tail(15),
    x="Количество_конкурентов",
    y="Страна",
    orientation="h",
    text="Количество_конкурентов",
    title="Топ стран по количеству конкурентов",
    color="Количество_конкурентов",
    color_continuous_scale="YlGnBu"
)
fig_countries.update_layout(
    height=550,
    title_x=0.5,
    coloraxis_showscale=False,
    xaxis_title="Количество конкурентов",
    yaxis_title="",
    template="plotly_white"
)
fig_countries.update_traces(
    textposition="outside"
)
fig_countries.show()

# популярные услуги
fig_services = px.bar(
    service_metrics_df.sort_values("Количество_конкурентов", ascending=True),
    x="Количество_конкурентов",
    y="Услуга",
    orientation="h",
    text="Количество_конкурентов",
    title="Какие услуги чаще всего предлагают конкуренты",
    color="Количество_конкурентов",
    color_continuous_scale="Purples"
)
fig_services.update_layout(
    height=500,
    title_x=0.5,
    coloraxis_showscale=False,
    xaxis_title="Количество конкурентов",
    yaxis_title="",
    template="plotly_white"
)
fig_services.update_traces(
    textposition="outside"
)
fig_services.show()

# объединенный график числа стран и услуг у конкурентов
position_df = pd.DataFrame(
    {
        "Конкурент": site_df["name"],
        "Число стран": site_df["countries_count"],
        "Число услуг": site_df["services_count"],
        "Объём текста": site_df["total_text_length"],
        "Сайт": site_df["website"]
    }
)

fig_position = px.scatter(
    position_df,
    x="Число стран",
    y="Число услуг",
    size="Объём текста",
    hover_name="Конкурент",
    hover_data={
        "Сайт": True,
        "Объём текста": True,
        "Число стран": True,
        "Число услуг": True
    },
    title="Позиционирование конкурентов: число стран × число услуг",
    color="Число стран",
    color_continuous_scale="Blues"
)
fig_position.update_layout(
    height=550,
    title_x=0.5,
    xaxis_title="Число найденных стран",
    yaxis_title="Число найденных услуг",
    template="plotly_white",
    coloraxis_showscale=False
)

fig_position.show()

# матрица услуг
service_heatmap = service_matrix_df.set_index("Конкурент")[list(services.keys())]

fig_service_heatmap = px.imshow(
    service_heatmap,
    title="Матрица услуг: конкурент × услуга",
    labels={"color": "Есть услуга"},
    color_continuous_scale=["#f2f2f2", "#4C78A8"],
    aspect="auto"
)

fig_service_heatmap.update_layout(
    height=650,
    title_x=0.5,
    template="plotly_white"
)

fig_service_heatmap.show()


# матрица стран (берем топ-15 по популярности)
top_countries = (
    country_metrics_df
    .sort_values("Количество_конкурентов", ascending=False)
    .head(15)["Страна"]
    .tolist()
)
country_heatmap = competitor_country_matrix_df.set_index("Конкурент")[top_countries]

fig_country_heatmap = px.imshow(
    country_heatmap,
    title="Матрица стран: конкурент × страна",
    labels={"color": "Страна найдена"},
    color_continuous_scale=["#f2f2f2", "#59A14F"],
    aspect="auto"
)
fig_country_heatmap.update_layout(
    height=700,
    title_x=0.5,
    template="plotly_white"
)

fig_country_heatmap.show()